In [6]:
pip install pandas numpy statsmodels scikit-learn scipy matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [7]:
import numpy as np
import pandas as pd
import json
import time
import requests
from datetime import date
from scipy.interpolate import CubicSpline

# Electricity Consumption Dataset

In [8]:
def generate_monthly_synthetic(file_path):
    # 1. Load your real annual dataset
    df = pd.read_csv(r"C:\Users\Asus\OneDrive\Desktop\Sem 10 September 2026\Final Year Project 2\dataset\annual_electricity_households.csv")
    df["date"] = pd.to_datetime(df["date"])

    states = df["state"].unique()
    all_synthetic_records = []

    for state in states:
        state_df = df[df["state"] == state].sort_values("date")

        # Extract real January snapshot coordinates
        annual_years = state_df["date"].dt.year.values
        real_counts = state_df["households"].values

        # Generate target continuous timeline (Every month from start year to June 2026)
        start_year = int(annual_years[0])
        target_months = pd.date_range(
            start=f"{start_year}-01-01", end="2026-06-01", freq="MS"
        )

        # 2. Fit a Cubic Spline anchored to the January real data points for smooth monthly transition
        cs = CubicSpline(annual_years, real_counts, extrapolate=True)
        monthly_values = cs(target_months.year + (target_months.month - 1) / 12.0)

        state_monthly_df = pd.DataFrame({
            "date": target_months,
            "state": state,
            "households_synthetic": monthly_values
        })

        # 3. Exact Anchor Hard-Lock: Force January of every year to match OpenDOSM real data 100%
        for index, row in state_df.iterrows():
            yr = row["date"].year
            jan_date = pd.Timestamp(year=yr, month=1, day=1)
            mask = state_monthly_df["date"] == jan_date
            if mask.any():
                state_monthly_df.loc[mask, "households_synthetic"] = row["households"]

        # Convert final monthly counts to integers
        state_monthly_df["households_synthetic"] = state_monthly_df[
            "households_synthetic"
        ].astype(int)

        all_synthetic_records.append(state_monthly_df)

    # Combine everything back into a clean dataset
    final_df = pd.concat(all_synthetic_records)
    return final_df[["date", "state", "households_synthetic"]]


if __name__ == "__main__":
    output_data = generate_monthly_synthetic("annual_electricity_households.csv")
    output_data.to_csv("monthly_households.csv", index=False)
    print("Successfully generated monthly household accessibility dataset from real annual records!")
    print(output_data.head(15))  # Display the first 15 months (Jan 2018 - Mar 2019)

Successfully generated monthly household accessibility dataset from real annual records!
         date     state  households_synthetic
0  2018-01-01  Malaysia               8452376
1  2018-02-01  Malaysia               8470523
2  2018-03-01  Malaysia               8488380
3  2018-04-01  Malaysia               8505968
4  2018-05-01  Malaysia               8523313
5  2018-06-01  Malaysia               8540438
6  2018-07-01  Malaysia               8557366
7  2018-08-01  Malaysia               8574122
8  2018-09-01  Malaysia               8590730
9  2018-10-01  Malaysia               8607212
10 2018-11-01  Malaysia               8623594
11 2018-12-01  Malaysia               8639898
12 2019-01-01  Malaysia               8656150
13 2019-02-01  Malaysia               8672371
14 2019-03-01  Malaysia               8688587


In [9]:
# 1. Read files with proper date parsing
elec_path = r"C:\Users\Asus\OneDrive\Desktop\Sem 10 September 2026\Final Year Project 2\dataset\monthly_electricity_consumption.csv"
hh_path = r"C:\Users\Asus\OneDrive\Desktop\Sem 10 September 2026\Final Year Project 2\dataset\monthly_households.csv"

elec = pd.read_csv(elec_path)
hh = pd.read_csv(hh_path)

# Clean column names in case there are hidden spaces
elec.columns = elec.columns.str.strip().str.lower()
hh.columns = hh.columns.str.strip().str.lower()

# Parse dates safely
elec["date"] = pd.to_datetime(elec["date"], dayfirst=True, errors="coerce")
hh["date"] = pd.to_datetime(hh["date"], errors="coerce")

# Automatically identify the numeric consumption column name
val_col = [c for c in elec.columns if c not in ["date", "sector"]][0]

# 2. Filter electricity sector
elec = elec[elec["sector"] == "local_domestic"][["date", val_col]]
elec = elec.rename(columns={val_col: "consumption"})

# 3. Filter households for Malaysia (already aggregated at country level)
hh_total = (
    hh[hh["state"] == "Malaysia"]
    .groupby("date", as_index=False)["households_synthetic"]
    .sum()
    .rename(columns={"households_synthetic": "households"})
)

# 4. Merge datasets on exact monthly date
df = pd.merge(elec, hh_total, on="date", how="inner")

# 5. Sort by date
df = df.sort_values("date")

# 6. Calculate kWh per household
df["kwh_per_household"] = df["consumption"] * 1_000_000 / df["households"]

# 7. Rename & Save
df = df.rename(columns={"consumption": "domestic_mkwh"})
df.to_csv("kwh_per_household.csv", index=False)

# Preview results
print(df.head(12))
print(df.tail(6))
print(df["kwh_per_household"].describe())

         date  domestic_mkwh  households  kwh_per_household
0  2018-01-01    2455.819450     8452376         290.547823
1  2018-02-01    2530.541863     8470523         298.746826
2  2018-03-01    2584.769732     8488380         304.506835
3  2018-04-01    2677.800560     8505968         314.814323
4  2018-05-01    2742.060788     8523313         321.713023
5  2018-06-01    2627.777866     8540438         307.686546
6  2018-07-01    2745.100789     8557366         320.788054
7  2018-08-01    2683.949408     8574122         313.029067
8  2018-09-01    2563.818575     8590730         298.440130
9  2018-10-01    2584.436869     8607212         300.264112
10 2018-11-01    2538.479838     8623594         294.364489
11 2018-12-01    2680.255015     8639898         310.218363
          date  domestic_mkwh  households  kwh_per_household
96  2026-01-01    3679.052434    10113547         363.774691
97  2026-02-01    3677.468606    10116866         363.498796
98  2026-03-01    4089.630784    1011

# Environmental Condition Dataset

In [ ]:
START, END = 2018, 2026

STATES = {
    "Johor": ("Johor Bahru", 1.4655, 103.7578),
    "Kedah": ("Alor Setar", 6.1248, 100.3678),
    "Kelantan": ("Kota Bharu", 6.1399, 102.2422),
    "Melaka": ("Bandaraya Melaka", 2.1960, 102.2405),
    "Negeri Sembilan": ("Seremban", 2.7297, 101.9381),
    "Pahang": ("Kuantan", 3.7634, 103.2202),
    "Perak": ("Ipoh", 4.5841, 101.0829),
    "Perlis": ("Kangar", 6.4455, 100.2002),
    "Pulau Pinang": ("George Town", 5.4112, 100.3354),
    "Sabah": ("Kota Kinabalu", 5.9804, 116.0735),
    "Sarawak": ("Kuching", 1.5531, 110.3450),
    "Selangor": ("Shah Alam", 3.0733, 101.5185),
    "Terengganu": ("Kuala Terengganu", 5.3302, 103.1408),
    "WP Kuala Lumpur": ("Kuala Lumpur", 3.1409, 101.6932),
    "WP Putrajaya": ("Putrajaya", 2.9264, 101.6964),
    "WP Labuan": ("Labuan", 5.2852, 115.2478),
}

URL = "https://power.larc.nasa.gov/api/temporal/monthly/point"
# Tambah 'WS2M' untuk Kelajuan Angin (Wind Speed at 2 meters)
PARAMS = ["T2M", "RH2M", "PRECTOTCORR", "WS2M"]

frames = []
for state, (city, lat, lon) in STATES.items():
    r = requests.get(URL, params={
        "parameters": ",".join(PARAMS),
        "community": "AG",
        "latitude": lat, "longitude": lon,
        "start": START, "end": END,
        "format": "JSON",
    }, timeout=60)
    r.raise_for_status()
    payload = r.json()

    safe = state.replace(" ", "_")
    with open(f"raw_nasa_{safe}.json", "w") as f:
        json.dump(payload, f)

    df = pd.DataFrame(payload["properties"]["parameter"])
    df.index.name = "ym"
    df = df.reset_index()
    df = df[~df["ym"].str.endswith("13")]             # buang purata tahunan
    df["date"] = pd.to_datetime(df["ym"], format="%Y%m")
    df = df.replace(-999, float("nan"))
    df.insert(0, "state", state)
    df.insert(1, "ibu_negeri", city)
    df.insert(2, "latitude", lat)
    df.insert(3, "longitude", lon)
    
    # Tukar nama kepada kolum yang lebih mudah dibaca termasuk angin
    df = df.rename(columns={
        "T2M": "temperature_C",
        "RH2M": "humidity_pct",
        "PRECTOTCORR": "rainfall_mm_per_day",
        "WS2M": "wind_speed_m_s",
    })
    
    df = df[["state", "ibu_negeri", "latitude", "longitude", "date",
             "temperature_C", "humidity_pct", "rainfall_mm_per_day", "wind_speed_m_s"]]
    frames.append(df)
    print("OK:", state, len(df), "rows")
    time.sleep(1)

# Gabungkan semua rekod negeri
allc = pd.concat(frames)

# Hadkan tarikh strictly dari Jan 2018 hingga Jun 2026
allc = allc[(allc["date"] >= "2018-01-01") & (allc["date"] <= "2026-06-01")]

# 1. Simpan semua negeri dalam satu fail (format panjang)
allc.to_csv("weather_all_states.csv", index=False)

# 2. Simpan purata bulanan Malaysia
avg = (allc.groupby("date")[["temperature_C", "humidity_pct", "rainfall_mm_per_day", "wind_speed_m_s"]]
           .mean().reset_index())
avg.to_csv("weather_monthly_average.csv", index=False)

print(avg.head(12))
print(allc.isna().sum())
print("Downloaded at:", date.today())

OK: Johor 106 rows
OK: Kedah 106 rows
OK: Kelantan 106 rows
OK: Melaka 106 rows
OK: Negeri Sembilan 106 rows
OK: Pahang 106 rows
OK: Perak 106 rows
OK: Perlis 106 rows
OK: Pulau Pinang 106 rows
OK: Sabah 106 rows
OK: Sarawak 106 rows
OK: Selangor 106 rows
OK: Terengganu 106 rows
OK: WP Kuala Lumpur 106 rows
OK: WP Putrajaya 106 rows
OK: WP Labuan 106 rows
         date  temperature_C  humidity_pct  rainfall_mm_per_day  \
0  2018-01-01      25.558125     87.002500            11.331250   
1  2018-02-01      25.692500     82.988125             3.522500   
2  2018-03-01      26.671250     82.744375             4.507500   
3  2018-04-01      27.206875     82.269375             5.653750   
4  2018-05-01      27.456875     82.698750             8.468750   
5  2018-06-01      27.039375     82.711875             6.781250   
6  2018-07-01      26.692500     84.015000             5.010625   
7  2018-08-01      26.830000     82.913125             4.176250   
8  2018-09-01      26.621875     82.360

: 